# Phase 3 - Analyse descriptive et visualisation du pipeline CNN-BSV

Objectifs :
- decrire le corpus BSV ;
- controler la construction du jeu experimental ;
- decrire les predictions CNN ;
- analyser la distribution des verdicts du moteur BSV ;
- identifier les limites avant l'evaluation statistique.

Ce notebook ne mesure pas encore la performance du moteur. La verite terrain
est utilisee uniquement pour decrire la composition du jeu experimental. Les
mesures de performance et les tests statistiques sont reserves au script
`src/phase3_evaluate.py`.

Pipeline :

```text
Corpus BSV
-> extraction des observations
-> conservation des 3 761 images avec verite separee
-> attribution experimentale des scenarios
-> inference CNN
-> requete moteur sans fuite de vraie classe
-> moteur de compatibilite
-> analyse descriptive
```

## Regles methodologiques

- Toujours distinguer le nombre d'images du nombre de predictions Top-K.
- `cnn_topk_predictions.json` est un fichier de suivi experimental.
- `requete_phase3.json` est la vraie entree du moteur.
- Les champs derives de la vraie classe (`statut_bsv`, `classe_cnn_source`,
  `bioagresseur_bsv`, `type_bioagresseur_bsv`) ne doivent pas etre presents
  dans la requete moteur.
- Les analyses de performance et les tests statistiques sont exclus de ce
  notebook.

In [ ]:
from __future__ import annotations

import json
import math
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'config' / 'pipeline.json').is_file():
            return candidate
    raise FileNotFoundError('Racine du projet introuvable depuis le notebook')

PROJECT_ROOT = find_project_root()

SRC_DIR = PROJECT_ROOT / "src" / "moteur"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

MOTEUR_IMPORT_OK = False
MOTEUR_IMPORT_ERROR = ""
try:
    import moteur_compatibilite as moteur
    MOTEUR_IMPORT_OK = True
except Exception as exc:
    moteur = None
    MOTEUR_IMPORT_ERROR = str(exc)
    print(f"Avertissement: import moteur_compatibilite impossible: {exc}")

DATA = PROJECT_ROOT / "data"
PROCESSED = DATA / "processed" / "moteur"
PHASE3 = PROCESSED / "phase3"
OUTPUTS = PROJECT_ROOT / "outputs" / "moteur" / "phase3"
RAW = DATA / "raw" / "moteur"

PATHS = {
    "corpus_manifest": PROCESSED / "bsv_corpus_manifest.csv",
    "bulletins": PROCESSED / "bsv_bulletins.csv",
    "observations": PROCESSED / "bsv_observations.csv",
    "synthese_tomate": PROCESSED / "synthese_tomate.csv",
    "synthese_bioagresseurs": PROCESSED / "synthese_bioagresseurs_tomate_par_region.csv",
    "periodes_region": PROCESSED / "periodes_bioagresseurs_tomate_par_region.csv",
    "periodes_annee": PROCESSED / "periodes_bioagresseurs_tomate_par_region_annee.csv",
    "ground_truth": PHASE3 / "ground_truth.csv",
    "manifest_masked": PHASE3 / "phase3_test_manifest_masked.csv",
    "metadata": PHASE3 / "phase3_test_with_metadata.csv",
    "assignments": PHASE3 / "phase3_scenario_assignments.csv",
    "coverage": PHASE3 / "phase3_bsv_metadata_coverage.csv",
    "cnn_predictions": OUTPUTS / "cnn_topk_predictions.json",
    "engine_query": OUTPUTS / "requete_phase3.json",
    "compatibility": PROCESSED / "bsv_compatibility_phase3.csv",
    "referential": RAW / "referentiel_cnn_bsv.csv",
    "analysis_full": OUTPUTS / "analysis" / "phase3_analysis_full.csv",
    "analysis_integrity": OUTPUTS / "analysis" / "phase3_integrity_checks.csv",
    "insufficiency_consistency": OUTPUTS / "analysis" / "phase3_insufficiency_consistency.csv",
}

ALLOWED_REGIONS = {"Grand Est", "Pays de la Loire", "Provence-Alpes-Côte d'Azur"}
ALLOWED_VERDICTS = {"Coherent", "Non_corrobore", "Indetermine"}
TRUTH_LEAKAGE_FIELDS = {"true_label", "scenario_assigne", "scenario_realise", "scenario_realisable", "cause_insuffisance", "classe_couverte_bsv", "context_id", "bulletin_id_assigne", "seed", "statut_bsv", "classe_cnn_source", "bioagresseur_bsv", "type_bioagresseur_bsv"}

In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df


def read_csv_auto(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Fichier manquant: {path}")
        return pd.DataFrame()
    try:
        return clean_columns(pd.read_csv(path, encoding="utf-8-sig", sep=None, engine="python"))
    except UnicodeDecodeError:
        return clean_columns(pd.read_csv(path, encoding="latin-1", sep=None, engine="python"))


def read_json_records(path: Path) -> list[dict]:
    if not path.exists():
        print(f"Fichier manquant: {path}")
        return []
    with path.open(encoding="utf-8-sig") as handle:
        payload = json.load(handle)
    return payload if isinstance(payload, list) else [payload]


def pct(n, d):
    return round(100 * n / d, 2) if d else 0.0


def value_table(df: pd.DataFrame, col: str, label: str | None = None) -> pd.DataFrame:
    if df.empty or col not in df.columns:
        return pd.DataFrame(columns=[label or col, "n", "pct"])
    out = df[col].fillna("(manquant)").value_counts(dropna=False).rename_axis(label or col).reset_index(name="n")
    out["pct"] = (100 * out["n"] / out["n"].sum()).round(2)
    return out


def bar_counts(series: pd.Series, title: str, top: int | None = None, horizontal: bool = False):
    counts = series.fillna("(manquant)").value_counts()
    if top:
        counts = counts.head(top)
    ax = counts.sort_values().plot(kind="barh" if horizontal else "bar")
    ax.set_title(title)
    ax.set_xlabel("Effectif" if horizontal else "")
    ax.set_ylabel("" if horizontal else "Effectif")
    plt.tight_layout()
    plt.show()


def stacked_percent(df: pd.DataFrame, index: str, columns: str, title: str):
    if df.empty or index not in df.columns or columns not in df.columns:
        print("Donnees insuffisantes.")
        return
    tab = pd.crosstab(df[index].fillna("(manquant)"), df[columns].fillna("(manquant)"), normalize="index") * 100
    ax = tab.plot(kind="bar", stacked=True, figsize=(12, 5))
    ax.set_title(title)
    ax.set_ylabel("%")
    ax.legend(title=columns, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


def heatmap_table(table: pd.DataFrame, title: str):
    if table.empty:
        print("Table vide.")
        return
    fig, ax = plt.subplots(figsize=(max(8, table.shape[1] * 1.0), max(4, table.shape[0] * 0.35)))
    im = ax.imshow(table.values, aspect="auto", cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(range(table.shape[1]))
    ax.set_xticklabels(table.columns, rotation=45, ha="right")
    ax.set_yticks(range(table.shape[0]))
    ax.set_yticklabels(table.index)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            value = table.iloc[i, j]
            if pd.notna(value) and value != 0:
                ax.text(j, i, str(int(value)), ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


def parse_dates(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

def resolve_project_path(value) -> Path:
    """Resout un chemin stocke dans un CSV, qu'il soit Windows, Linux ou relatif."""
    if pd.isna(value):
        return PROJECT_ROOT / "__chemin_manquant__"
    raw = str(value).strip().strip('"').replace("\\", "/")
    p = Path(raw)
    if p.is_absolute():
        return p
    if raw.startswith("/mnt/") and len(raw) > 6 and raw[5].isalpha():
        drive = raw[5].upper()
        rest = raw[6:].lstrip("/")
        return Path(f"{drive}:/" + rest)
    return PROJECT_ROOT / p


## 1. Chargement et controle des fichiers

In [ ]:
tables = {name: read_csv_auto(path) for name, path in PATHS.items() if path.suffix.lower() == ".csv"}
cnn_records = read_json_records(PATHS["cnn_predictions"])
query_records = read_json_records(PATHS["engine_query"])

corpus = tables.get("corpus_manifest", pd.DataFrame())
bulletins = parse_dates(tables.get("bulletins", pd.DataFrame()), ["date_bulletin"])
observations = tables.get("observations", pd.DataFrame())
synthese_tomate = parse_dates(tables.get("synthese_tomate", pd.DataFrame()), ["date_bulletin"])
periodes_region = parse_dates(tables.get("periodes_region", pd.DataFrame()), ["premiere_date_bulletin", "derniere_date_bulletin"])
periodes_annee = parse_dates(tables.get("periodes_annee", pd.DataFrame()), ["premiere_date_bulletin", "derniere_date_bulletin"])
ground_truth = tables.get("ground_truth", pd.DataFrame())
manifest_masked = tables.get("manifest_masked", pd.DataFrame())
metadata = parse_dates(tables.get("metadata", pd.DataFrame()), ["date_observation"])
assignments = parse_dates(tables.get("assignments", pd.DataFrame()), ["date_observation"])
if not assignments.empty:
    experimental = assignments.rename(columns={"true_label": "classe_cnn_source", "scenario_realise": "statut_bsv"})
    metadata = metadata.merge(
        experimental[["observation_id", "classe_cnn_source", "statut_bsv", "scenario_assigne", "scenario_realisable", "raison_non_realisation", "classe_couverte_bsv", "context_id", "bulletin_id_assigne"]],
        on="observation_id", how="left", validate="one_to_one"
    )
    metadata["type_bioagresseur_bsv"] = "non_applicable_v3"
    rejected = assignments.loc[~assignments["scenario_realisable"]].rename(columns={"raison_non_realisation": "raison_rejet"}).copy()
    coverage = assignments.groupby("true_label", as_index=False).agg(
        nb_images_source=("observation_id", "size"),
        classe_conservee=("classe_couverte_bsv", "max"),
    ).rename(columns={"true_label": "classe_cnn"})
    coverage["nb_fenetres_bsv"] = 0
    coverage["bioagresseurs_bsv"] = ""
    coverage["regions_bsv"] = ""
    coverage["annees_bsv"] = ""
else:
    rejected = tables.get("rejected", pd.DataFrame())
    coverage = tables.get("coverage", pd.DataFrame())
compat = parse_dates(tables.get("compatibility", pd.DataFrame()), ["date_observation"])
referential = tables.get("referential", pd.DataFrame())

def id_column_for(name: str, df: pd.DataFrame) -> str | None:
    candidates = {
        "corpus_manifest": ["relative_path", "md5"],
        "bulletins": ["bulletin_id"],
        "observations": ["bulletin_id"],
        "synthese_tomate": ["bulletin_id"],
        "ground_truth": ["observation_id"],
        "manifest_masked": ["observation_id"],
        "metadata": ["observation_id"],
        "rejected": ["observation_id"],
        "coverage": ["classe_cnn"],
        "compatibility": ["image_id"],
        "referential": ["classe_dataset"],
    }.get(name, [])
    for col in candidates:
        if col in df.columns:
            return col
    return df.columns[0] if len(df.columns) else None

summary_rows = []
for name, df in tables.items():
    id_col = id_column_for(name, df)
    summary_rows.append({
        "fichier": name,
        "lignes": len(df),
        "colonnes": len(df.columns),
        "colonne_id": id_col or "",
        "identifiants_uniques": df[id_col].nunique(dropna=True) if id_col else 0,
        "doublons_lignes": int(df.duplicated().sum()) if not df.empty else 0,
        "cellules_manquantes_pct": round(100 * df.isna().sum().sum() / (df.shape[0] * max(df.shape[1], 1)), 2) if not df.empty else 0,
    })
display(pd.DataFrame(summary_rows))

unit_rows = [
    {"Fichier": "Bulletins", "Unite statistique": "Un bulletin", "Lignes": len(bulletins), "Identifiants uniques": bulletins["bulletin_id"].nunique() if "bulletin_id" in bulletins else 0},
    {"Fichier": "Observations", "Unite statistique": "Une observation BSV", "Lignes": len(observations), "Identifiants uniques": observations["bulletin_id"].nunique() if "bulletin_id" in observations else 0},
    {"Fichier": "Metadonnees phase 3", "Unite statistique": "Une image", "Lignes": len(metadata), "Identifiants uniques": metadata["observation_id"].nunique() if "observation_id" in metadata else 0},
    {"Fichier": "Compatibilite", "Unite statistique": "Une prediction Top-K", "Lignes": len(compat), "Identifiants uniques": compat["image_id"].nunique() if "image_id" in compat else 0},
]
display(pd.DataFrame(unit_rows))
print(f"Predictions CNN JSON: {len(cnn_records)} images")
print(f"Requetes moteur JSON: {len(query_records)} images")

## 1.1 Controle de l'import du moteur et des fenetres litterature

Cette cellule rend explicite l'etat de l'import de `moteur_compatibilite`. Si l'import echoue, les controles sur les fenetres litterature ne doivent pas etre interpretes comme des resultats biologiques.

In [ ]:
moteur_status = pd.DataFrame([
    {
        "controle": "Import moteur_compatibilite",
        "resultat": "OK" if MOTEUR_IMPORT_OK else "ECHEC",
        "detail": "" if MOTEUR_IMPORT_OK else MOTEUR_IMPORT_ERROR,
    },
    {
        "controle": "Fenetres litterature disponibles",
        "resultat": bool(moteur is not None and hasattr(moteur, "LITERATURE_WINDOWS")),
        "detail": len(getattr(moteur, "LITERATURE_WINDOWS", {})) if moteur is not None else 0,
    },
])
display(moteur_status)

if moteur is None or not hasattr(moteur, "LITERATURE_WINDOWS"):
    print("ALERTE: les fenetres litterature ne sont pas disponibles ; la section correspondante sera non interpretable.")

# Partie A - Description du corpus BSV

In [ ]:
if not corpus.empty:
    total_pdf = len(corpus)
    exploitable = int(corpus.get("corpus_exploitable", pd.Series(dtype=bool)).astype(str).str.lower().isin(["true", "1", "oui"]).sum()) if "corpus_exploitable" in corpus else 0
    duplicates = int(corpus.get("duplicate_exact", pd.Series(dtype=bool)).astype(str).str.lower().isin(["true", "1", "oui"]).sum()) if "duplicate_exact" in corpus else 0
    display(pd.DataFrame([
        {"indicateur": "PDF collectes", "valeur": total_pdf},
        {"indicateur": "PDF exploitables", "valeur": exploitable},
        {"indicateur": "Documents rejetes", "valeur": total_pdf - exploitable},
        {"indicateur": "Taux exploitabilite (%)", "valeur": pct(exploitable, total_pdf)},
        {"indicateur": "Doublons exacts detectes", "valeur": duplicates},
        {"indicateur": "Bulletins structures", "valeur": len(bulletins)},
    ]))
    if "corpus_exploitable" in corpus:
        bar_counts(corpus["corpus_exploitable"].astype(str), "Documents par statut d'exploitabilite")

if not bulletins.empty:
    bar_counts(bulletins["region"], "Bulletins par region")
    bt = bulletins.dropna(subset=["date_bulletin"]).assign(
        annee=lambda d: d["date_bulletin"].dt.year,
        mois=lambda d: d["date_bulletin"].dt.month,
    )
    display(pd.crosstab(bt["region"], bt["annee"], margins=True))
    heatmap_table(bt.groupby(["annee", "mois"]).size().unstack(fill_value=0), "Bulletins par annee et mois")
    display(bt.groupby("region").agg(premiere_date=("date_bulletin", "min"), derniere_date=("date_bulletin", "max"), nb_bulletins=("bulletin_id", "nunique")))

    delays = []
    for region, group in bt.sort_values("date_bulletin").groupby("region"):
        for value in group["date_bulletin"].diff().dt.days.dropna():
            delays.append({"region": region, "delai_jours": value})
    delays_df = pd.DataFrame(delays)
    if not delays_df.empty:
        display(delays_df.groupby("region")["delai_jours"].describe().round(2))
        delays_df["delai_jours"].plot(kind="hist", bins=20, title="Delais entre deux bulletins consecutifs")
        plt.xlabel("Jours")
        plt.tight_layout()
        plt.show()

In [ ]:
obs_metrics = [{"indicateur": "Observations totales", "valeur": len(observations)}]
if "culture" in observations:
    culture_norm = observations["culture"].fillna("").str.lower()
    obs_metrics += [
        {"indicateur": "Observations tomate", "valeur": int(culture_norm.str.contains("tomate", na=False).sum())},
        {"indicateur": "Observations solanacees", "valeur": int(culture_norm.str.contains("solan", na=False).sum())},
    ]
if "bioagresseur" in observations:
    obs_metrics.append({"indicateur": "Libelles sanitaires distincts extraits", "valeur": observations["bioagresseur"].nunique(dropna=True)})
obs_metrics.append({"indicateur": "Lignes synthese tomate/solanacees", "valeur": len(synthese_tomate)})
display(pd.DataFrame(obs_metrics))

if "type" in observations:
    display(value_table(observations, "type"))
    bar_counts(observations["type"], "Observations par type")
if not synthese_tomate.empty:
    bar_counts(synthese_tomate["region"], "Observations tomate/solanacees par region")
    bar_counts(synthese_tomate["bioagresseur"], "Principaux bioagresseurs tomate/solanacees", top=20, horizontal=True)
    st = synthese_tomate.dropna(subset=["date_bulletin"]).assign(mois=lambda d: d["date_bulletin"].dt.month)
    bar_counts(st["mois"].astype(str), "Observations tomate/solanacees par mois")

In [ ]:
if not periodes_region.empty and {"region", "bioagresseur", "nb_bulletins"} <= set(periodes_region.columns):
    bio_region = periodes_region.pivot_table(index="bioagresseur", columns="region", values="nb_bulletins", aggfunc="sum", fill_value=0)
    bio_region["Total"] = bio_region.sum(axis=1)
    bio_region = bio_region.sort_values("Total", ascending=False)
    display(bio_region)
    heatmap_table(bio_region.drop(columns=["Total"]), "Mentions par bioagresseur et region")
    region_cols = [c for c in bio_region.columns if c != "Total"]
    print("Bioagresseurs communs aux regions:", bio_region[(bio_region[region_cols] > 0).all(axis=1)].index.tolist())
    print("Bioagresseurs propres a une region:", bio_region[(bio_region[region_cols] > 0).sum(axis=1).eq(1)].index.tolist())

if not periodes_annee.empty:
    display(periodes_annee.groupby("bioagresseur")["annee"].nunique().sort_values(ascending=False).reset_index(name="nb_annees_couverture"))

In [ ]:
coverage_rows = []

coverage_by_class = (
    coverage.set_index("classe_cnn")
    if not coverage.empty and "classe_cnn" in coverage.columns
    else pd.DataFrame()
)

for _, row in referential.iterrows() if not referential.empty else []:
    cls = row.get("classe_dataset", "")
    cov = coverage_by_class.loc[cls] if cls in coverage_by_class.index else pd.Series(dtype=object)
    conserved = bool(cov.get("classe_conservee", False)) if not cov.empty else False
    nb_windows = int(pd.to_numeric(pd.Series([cov.get("nb_fenetres_bsv", 0)]), errors="coerce").fillna(0).iloc[0]) if not cov.empty else 0

    if cls == "tomato_leaf_blight":
        status = "classe_ambigue"
    elif conserved:
        status = "couvert_selection_phase3"
    elif nb_windows > 0:
        status = "couvert_corpus_non_retenu"
    else:
        status = "non_couvert_selection_phase3"

    coverage_rows.append({
        "classe_cnn": cls,
        "nom_fr": row.get("nom_fr", ""),
        "agent_referentiel": row.get("agent", ""),
        "type": row.get("type", ""),
        "couverture_bsv": status,
        "classe_conservee": conserved,
        "bioagresseurs_bsv": cov.get("bioagresseurs_bsv", "") if not cov.empty else "",
        "regions_bsv": cov.get("regions_bsv", "") if not cov.empty else "",
        "annees_bsv": cov.get("annees_bsv", "") if not cov.empty else "",
        "nb_fenetres_bsv": nb_windows,
    })

cnn_bsv_coverage = pd.DataFrame(coverage_rows)
display(cnn_bsv_coverage.sort_values(["couverture_bsv", "nb_fenetres_bsv"], ascending=[True, False]))
if not cnn_bsv_coverage.empty:
    bar_counts(cnn_bsv_coverage.set_index("classe_cnn")["nb_fenetres_bsv"], "Fenetres BSV par classe CNN selon le referentiel", horizontal=True)
if not coverage.empty:
    display(coverage)


# Partie B - Construction du jeu experimental

In [ ]:
initial_images = ground_truth["observation_id"].nunique() if "observation_id" in ground_truth else 0
masked_images = manifest_masked["observation_id"].nunique() if "observation_id" in manifest_masked else 0
retained_images = metadata["observation_id"].nunique() if "observation_id" in metadata else 0
rejected_images = rejected["observation_id"].nunique() if "observation_id" in rejected else 0
cnn_ok = sum(bool(r.get("inference_ok")) for r in cnn_records)
engine_images = compat["image_id"].nunique() if "image_id" in compat else 0
hors_saison = int((metadata["statut_bsv"] == "insuffisant").sum()) if "statut_bsv" in metadata else 0

funnel = pd.DataFrame([
    {"etape": "Test initial", "nombre_images": initial_images},
    {"etape": "Images anonymisees", "nombre_images": masked_images},
    {"etape": "Images retenues", "nombre_images": retained_images},
    {"etape": "Contextes insuffisants", "nombre_images": hors_saison},
    {"etape": "Images rejetees", "nombre_images": rejected_images},
    {"etape": "Inference CNN reussie", "nombre_images": cnn_ok},
    {"etape": "Images avec sortie moteur", "nombre_images": engine_images},
])
funnel["pct_total_initial"] = funnel["nombre_images"].apply(lambda x: pct(x, initial_images))
display(funnel)
funnel.set_index("etape")["nombre_images"].plot(kind="bar", title="Entonnoir de selection des images")
plt.ylabel("Images")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

id_sets = {
    "ground_truth": set(ground_truth.get("observation_id", [])),
    "manifest_masked": set(manifest_masked.get("observation_id", [])),
    "metadata": set(metadata.get("observation_id", [])),
    "cnn_ok": {r.get("observation_id") for r in cnn_records if r.get("inference_ok")},
    "engine": set(compat.get("image_id", [])),
}
display(pd.DataFrame([
    {"controle": f"{left} inclus dans {right}", "manquants": len(id_sets[left] - id_sets[right])}
    for left, right in [("manifest_masked", "ground_truth"), ("metadata", "manifest_masked"), ("cnn_ok", "metadata"), ("engine", "cnn_ok")]
]))

In [ ]:
if not rejected.empty:
    display(value_table(rejected, "raison_rejet"))
    bar_counts(rejected["raison_rejet"], "Raisons de rejet")
    if "true_label" in rejected:
        bar_counts(rejected["true_label"], "Rejets par classe reelle", top=20, horizontal=True)
    if {"true_label", "raison_rejet"} <= set(rejected.columns):
        display(pd.crosstab(rejected["true_label"], rejected["raison_rejet"]))

In [ ]:
if not metadata.empty:
    meta = metadata.copy()
    meta["annee"] = meta["date_observation"].dt.year
    meta["mois"] = meta["date_observation"].dt.month
    for col, title in [
        ("classe_cnn_source", "Images retenues par classe source"),
        ("region", "Images retenues par region"),
        ("annee", "Images retenues par annee"),
        ("mois", "Images retenues par mois"),
        ("type_bioagresseur_bsv", "Images retenues par type"),
        ("statut_bsv", "Images retenues par statut BSV"),
    ]:
        if col in meta:
            display(value_table(meta, col))
            bar_counts(meta[col].astype(str), title, horizontal=(col == "classe_cnn_source"))
    heatmap_table(pd.crosstab(meta["classe_cnn_source"], meta["region"]), "Images retenues: classe x region")
    heatmap_table(pd.crosstab(meta["classe_cnn_source"], meta["annee"]), "Images retenues: classe x annee")

    counts = meta["classe_cnn_source"].value_counts()
    alerts = []
    for cls, n in counts.items():
        if n / len(meta) > 0.20:
            alerts.append({"alerte": "Classe > 20 % du jeu", "valeur": cls, "effectif": int(n)})
        if n < 30:
            alerts.append({"alerte": "Classe < 30 observations", "valeur": cls, "effectif": int(n)})
    for region, n in meta["region"].value_counts().items():
        if n / len(meta) > 0.50:
            alerts.append({"alerte": "Region > 50 % du jeu", "valeur": region, "effectif": int(n)})
    display(pd.DataFrame(alerts) if alerts else pd.DataFrame([{"alerte": "Aucune alerte descriptive majeure"}]))

## 10.1 Biais de selection : pool initial vs images retenues

Le funnel indique combien d'images sont conservees, mais pas comment la distribution des classes est modifiee. Cette cellule compare le pool initial (`ground_truth.csv`) et le jeu retenu (`phase3_test_with_metadata.csv`) afin de documenter le biais de selection introduit par la couverture BSV.

In [ ]:
if not ground_truth.empty and not metadata.empty and {'observation_id', 'true_label'} <= set(ground_truth.columns) and 'classe_cnn_source' in metadata.columns:
    initial_counts = ground_truth['true_label'].value_counts().rename('n_initial')
    retained_counts = metadata['classe_cnn_source'].value_counts().rename('n_retenu')
    selection_bias = pd.concat([initial_counts, retained_counts], axis=1).fillna(0).astype(int)
    selection_bias['pct_initial'] = (100 * selection_bias['n_initial'] / selection_bias['n_initial'].sum()).round(2)
    selection_bias['pct_retenu'] = (100 * selection_bias['n_retenu'] / selection_bias['n_retenu'].sum()).round(2)
    selection_bias['difference_points_pct'] = (selection_bias['pct_retenu'] - selection_bias['pct_initial']).round(2)
    selection_bias['taux_selection_pct'] = (100 * selection_bias['n_retenu'] / selection_bias['n_initial'].replace(0, np.nan)).round(2)
    selection_bias = selection_bias.sort_values('difference_points_pct', ascending=False)
    display(selection_bias)

    ax = selection_bias[['pct_initial', 'pct_retenu']].sort_values('pct_retenu').plot(kind='barh', figsize=(11, 7))
    ax.set_title('Distribution des classes : pool initial vs images retenues')
    ax.set_xlabel('% des images')
    plt.tight_layout()
    plt.show()

    display(selection_bias.loc[selection_bias['n_retenu'].eq(0), ['n_initial', 'pct_initial']].rename_axis('classes_absentes_apres_selection'))
else:
    print('Donnees insuffisantes pour comparer le biais de selection.')

In [ ]:
control_rows = []
if not metadata.empty:
    missing_images = sum(not resolve_project_path(p).exists() for p in metadata.get("image_path", []))
    control_rows = [
        {"controle": "Dates valides", "resultat": metadata["date_observation"].notna().all()},
        {"controle": "Regions autorisees", "resultat": metadata["region"].isin(ALLOWED_REGIONS).all()},
        {"controle": "Dates futures", "resultat": int((metadata["date_observation"] > pd.Timestamp.today()).sum())},
        {"controle": "Doublons observation_id", "resultat": int(metadata["observation_id"].duplicated().sum())},
        {"controle": "Images selectionnees introuvables apres resolution du chemin", "resultat": missing_images},
    ]
display(pd.DataFrame(control_rows))

# Partie C - Description des predictions CNN

In [ ]:
cnn_flat = []
for rec in cnn_records:
    preds = rec.get("predictions") or rec.get("top_k") or []
    for i, pred in enumerate(preds, start=1):
        cnn_flat.append({
            "observation_id": rec.get("observation_id") or rec.get("image_id"),
            "date_observation": rec.get("date_observation"),
            "region": rec.get("region"),
            "inference_ok": rec.get("inference_ok"),
            "rank": pred.get("rank", i),
            "label": pred.get("label") or pred.get("classe") or pred.get("class"),
            "score": pred.get("score"),
        })
cnn_df = pd.DataFrame(cnn_flat)
if not cnn_df.empty:
    cnn_df["score"] = pd.to_numeric(cnn_df["score"], errors="coerce")
    display(pd.DataFrame([
        {"indicateur": "Images fournies au CNN", "valeur": len(cnn_records)},
        {"indicateur": "Images avec prediction", "valeur": cnn_df["observation_id"].nunique()},
        {"indicateur": "Echecs inference", "valeur": sum(not bool(r.get("inference_ok")) for r in cnn_records)},
        {"indicateur": "Taux reussite inference (%)", "valeur": pct(sum(bool(r.get("inference_ok")) for r in cnn_records), len(cnn_records))},
        {"indicateur": "Predictions Top-K", "valeur": len(cnn_df)},
        {"indicateur": "Scores invalides/manquants", "valeur": int((cnn_df["score"].isna() | ~cnn_df["score"].between(0, 1)).sum())},
    ]))
    display(cnn_df.groupby("observation_id")["rank"].count().describe().to_frame("nb_predictions_par_image"))
    top1_cnn = cnn_df.sort_values(["observation_id", "rank", "score"], ascending=[True, True, False]).groupby("observation_id").head(1)
    display(value_table(top1_cnn, "label", "classe_top1"))
    bar_counts(top1_cnn["label"], "Predictions CNN Top-1 par classe", horizontal=True)
    rank_class = pd.crosstab(cnn_df["label"], cnn_df["rank"])
    heatmap_table(rank_class, "Classe predite x rang Top-K")

In [ ]:
if not cnn_df.empty:
    display(top1_cnn["score"].describe().to_frame("score_top1").round(4))
    top1_cnn["score"].plot(kind="hist", bins=30, title="Distribution des scores Top-1")
    plt.xlabel("Score")
    plt.tight_layout()
    plt.show()

    score_bins = pd.cut(top1_cnn["score"], bins=[0, 0.5, 0.7, 0.9, 1.01], right=False, include_lowest=True)
    display(score_bins.value_counts().sort_index().rename_axis("tranche_score").reset_index(name="n"))

    score_pivot = cnn_df.sort_values(["observation_id", "rank"]).pivot_table(index="observation_id", columns="rank", values="score", aggfunc="first")
    if {1, 2} <= set(score_pivot.columns):
        margins = (score_pivot[1] - score_pivot[2]).dropna()
        display(margins.describe().to_frame("marge_top1_top2").round(4))
        margins.plot(kind="hist", bins=30, title="Marge CNN Top-1 - Top-2")
        plt.xlabel("Marge")
        plt.tight_layout()
        plt.show()

    pairs = []
    for _, group in cnn_df.groupby("observation_id"):
        labels = group.sort_values("rank")["label"].dropna().tolist()
        for i in range(len(labels)):
            for j in range(i + 1, len(labels)):
                pairs.append(tuple(sorted([labels[i], labels[j]])))
    display(pd.DataFrame([{"classe_a": a, "classe_b": b, "n": n} for (a, b), n in Counter(pairs).most_common(30)]))

# Partie D - Description des sorties du moteur BSV

In [ ]:
compat_control = []
if not compat.empty:
    bad_verdicts = sorted(set(compat["compatibilite"].dropna()) - ALLOWED_VERDICTS)
    leak_cols = sorted(set(compat.columns) & TRUTH_LEAKAGE_FIELDS)
    compat_control = [
        {"controle": "Images uniques", "resultat": compat["image_id"].nunique(), "statut": "INFO"},
        {"controle": "Lignes predictions Top-K", "resultat": len(compat), "statut": "INFO"},
        {"controle": "Valeurs de verdict autorisees", "resultat": bad_verdicts, "statut": "OK" if not bad_verdicts else "Alerte"},
        {"controle": "Scores compris entre 0 et 1", "resultat": compat["score_cnn"].between(0, 1).all(), "statut": "OK" if compat["score_cnn"].between(0, 1).all() else "Alerte"},
        {"controle": "Absence de vraie classe dans sortie moteur", "resultat": leak_cols, "statut": "OK" if not leak_cols else "Alerte"},
    ]
    display(pd.DataFrame(compat_control))
    missing_table = compat.isna().mean().mul(100).round(2).rename("pct_manquant").reset_index()
    missing_table = missing_table.rename(columns={"index": "colonne"})
    display(missing_table)

In [ ]:
if not compat.empty:
    display(value_table(compat, "compatibilite", "verdict"))
    bar_counts(compat["compatibilite"], "Verdicts moteur - toutes predictions Top-K")
    decidable = compat["compatibilite"].isin(["Coherent", "Non_corrobore"]).sum()
    display(pd.DataFrame([
        {"indicateur": "Taux decidabilite Top-K (%)", "valeur": pct(decidable, len(compat))},
        {"indicateur": "Taux indetermination Top-K (%)", "valeur": pct((compat["compatibilite"] == "Indetermine").sum(), len(compat))},
    ]))

    query_top1 = pd.DataFrame([
        {"image_id": item.get("image_id", item.get("observation_id")), "classe_top1_requete": (item.get("top_k") or item.get("predictions"))[0].get("classe", (item.get("top_k") or item.get("predictions"))[0].get("label"))}
        for item in query_records if (item.get("top_k") or item.get("predictions"))
    ])
    top1_engine = query_top1.merge(compat, left_on=["image_id", "classe_top1_requete"], right_on=["image_id", "classe_cnn_predite"], how="left", validate="one_to_one")
    display(value_table(top1_engine, "compatibilite", "verdict_top1"))
    bar_counts(top1_engine["compatibilite"], "Verdicts moteur - Top-1 image")

    compat_ranked = compat.sort_values(["image_id", "score_cnn"], ascending=[True, False]).copy()
    compat_ranked["rang_inferre"] = compat_ranked.groupby("image_id").cumcount() + 1
    display(pd.crosstab(compat_ranked["rang_inferre"], compat_ranked["compatibilite"]))
    stacked_percent(compat_ranked, "rang_inferre", "compatibilite", "Verdicts par rang Top-K infere")

## 17.1 Scores CNN Top-1 selon le verdict moteur

Cette analyse reste descriptive : elle ne teste pas la calibration du CNN. Elle permet seulement de voir si les verdicts `Coherent`, `Non_corrobore` et `Indetermine` sont associes a des niveaux de confiance CNN differents.

In [ ]:
if 'top1_engine' in globals() and not top1_engine.empty and {'compatibilite', 'score_cnn'} <= set(top1_engine.columns):
    score_by_verdict = top1_engine.groupby('compatibilite')['score_cnn'].describe().round(4)
    display(score_by_verdict)

    ax = top1_engine.boxplot(column='score_cnn', by='compatibilite', grid=True, figsize=(8, 5))
    ax.set_title('Score CNN Top-1 par verdict moteur')
    ax.set_xlabel('Verdict moteur')
    ax.set_ylabel('Score CNN Top-1')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

    low_margin_note = top1_engine.assign(score_tranche=pd.cut(top1_engine['score_cnn'], bins=[0, 0.5, 0.7, 0.9, 1.01], right=False, include_lowest=True))
    display(pd.crosstab(low_margin_note['score_tranche'], low_margin_note['compatibilite'], normalize='index').mul(100).round(2))
else:
    print('Top-1 moteur indisponible pour croiser score CNN et compatibilite.')

In [ ]:
if not compat.empty:
    display(value_table(compat, "niveau_preuve"))
    display(pd.crosstab(compat["compatibilite"], compat["niveau_preuve"], margins=True))
    bar_counts(compat["niveau_preuve"], "Niveaux de preuve")
    ind = compat[compat["compatibilite"] == "Indetermine"]
    noncorr = compat[compat["compatibilite"] == "Non_corrobore"]
    if not ind.empty:
        bar_counts(ind["decision_reason"], "Principales raisons Indetermine", top=15, horizontal=True)
    if not noncorr.empty:
        bar_counts(noncorr["decision_reason"], "Principales raisons Non_corrobore", top=15, horizontal=True)

    class_summary = pd.crosstab(compat["classe_cnn_predite"], compat["compatibilite"])
    for col in ALLOWED_VERDICTS:
        if col not in class_summary.columns:
            class_summary[col] = 0
    class_summary["N"] = class_summary.sum(axis=1)
    class_summary["pct_coherent"] = (100 * class_summary["Coherent"] / class_summary["N"]).round(2)
    class_summary["pct_non_corrobore"] = (100 * class_summary["Non_corrobore"] / class_summary["N"]).round(2)
    class_summary["pct_indetermine"] = (100 * class_summary["Indetermine"] / class_summary["N"]).round(2)
    class_summary["decidabilite"] = (100 * (class_summary["Coherent"] + class_summary["Non_corrobore"]) / class_summary["N"]).round(2)
    display(class_summary.sort_values("pct_indetermine", ascending=False))
    stacked_percent(compat, "classe_cnn_predite", "compatibilite", "Verdicts par classe predite")

In [ ]:
if not compat.empty:
    comp_time = compat.copy()
    comp_time["annee"] = comp_time["date_observation"].dt.year
    comp_time["mois"] = comp_time["date_observation"].dt.month
    for col in ["region", "annee", "mois"]:
        display(pd.crosstab(comp_time[col], comp_time["compatibilite"], margins=True))
        stacked_percent(comp_time, col, "compatibilite", f"Verdicts par {col}")
    heatmap_table(pd.crosstab(comp_time["region"], comp_time["compatibilite"]), "Region x verdict")

In [ ]:
def compute_latest_bulletin_delay(row):
    if bulletins.empty or "date_bulletin" not in bulletins or "region" not in bulletins:
        return np.nan
    candidates = bulletins[(bulletins["region"] == row.get("region")) & (bulletins["date_bulletin"] <= row.get("date_observation"))]
    if candidates.empty:
        return np.nan
    delay = (row.get("date_observation") - candidates["date_bulletin"].max()).days
    return delay if delay <= 14 else np.nan

if not compat.empty:
    comp_delay = compat.copy()
    comp_delay["delai_bulletin_jours"] = comp_delay.apply(compute_latest_bulletin_delay, axis=1)
    display(comp_delay["delai_bulletin_jours"].describe().to_frame("delai_bulletin_jours").round(2))
    comp_delay["delai_bulletin_jours"].dropna().plot(kind="hist", bins=15, title="Delai entre observation et bulletin retenable")
    plt.xlabel("Jours")
    plt.tight_layout()
    plt.show()
    comp_delay["tranche_delai"] = pd.cut(comp_delay["delai_bulletin_jours"], bins=[-1, 3, 7, 10, 14], labels=["0-3 jours", "4-7 jours", "8-10 jours", "11-14 jours"])
    comp_delay["tranche_delai"] = comp_delay["tranche_delai"].astype("object").fillna("sans bulletin")
    stacked_percent(comp_delay, "tranche_delai", "compatibilite", "Verdicts par tranche de delai bulletin")

In [ ]:
if not compat.empty:
    compat_signal = compat.copy()
    compat_signal["type_signal"] = compat_signal["niveau_preuve"].fillna("autre")
    display(value_table(compat_signal, "type_signal"))
    display(pd.crosstab(compat_signal["type_signal"], compat_signal["compatibilite"], margins=True))
    compat_signal["niveau_risque_clean"] = compat_signal["niveau_risque"].fillna("non_renseigne").replace("", "non_renseigne")
    display(value_table(compat_signal, "niveau_risque_clean", "niveau_risque"))
    stacked_percent(compat_signal, "niveau_risque_clean", "compatibilite", "Verdicts selon le niveau de risque BSV")

In [ ]:
def lit_window_label(classe):
    if classe == "tomato_healthy":
        return "sans fenetre litterature"
    if moteur is None or not hasattr(moteur, "LITERATURE_WINDOWS"):
        return "non disponible"
    spec = moteur.LITERATURE_WINDOWS.get(classe, {}).get("plausible")
    if not spec:
        return "non definissable"
    return f"{spec[0][0]:02d}/{spec[0][1]:02d} - {spec[1][0]:02d}/{spec[1][1]:02d}"

if not compat.empty:
    lit = compat.copy()
    lit["fenetre_litterature"] = lit["classe_cnn_predite"].apply(lit_window_label)
    lit["hors_fenetre_litterature"] = lit["niveau_preuve"].eq("hors_periode_litterature")
    lit_summary = lit.groupby(["classe_cnn_predite", "fenetre_litterature"]).agg(n=("image_id", "count"), n_hors_fenetre=("hors_fenetre_litterature", "sum")).reset_index()
    lit_summary["pct_hors_fenetre"] = (100 * lit_summary["n_hors_fenetre"] / lit_summary["n"]).round(2)
    display(lit_summary.sort_values("pct_hors_fenetre", ascending=False))
    display(pd.crosstab(lit["hors_fenetre_litterature"], lit["compatibilite"]))

    hpred = compat[compat["classe_cnn_predite"] == "tomato_healthy"]
    display(pd.DataFrame([
        {"indicateur": "Images source saines retenues", "valeur": int((metadata["classe_cnn_source"] == "tomato_healthy").sum()) if "classe_cnn_source" in metadata else 0},
        {"indicateur": "Predictions moteur tomato_healthy", "valeur": len(hpred)},
        {"indicateur": "Images avec Top-K healthy", "valeur": hpred["image_id"].nunique()},
    ]))
    if not hpred.empty:
        display(pd.crosstab(hpred["compatibilite"], hpred["niveau_preuve"], margins=True))

# Partie E - Controles de coherence et exemples

In [ ]:
checks = []

def add_check(name, result, ok=True):
    checks.append({"controle": name, "resultat": result, "statut": "OK" if ok else "Alerte"})

add_check("Identifiants uniques dans le manifeste", int(manifest_masked["observation_id"].duplicated().sum()) if "observation_id" in manifest_masked else "NA", manifest_masked.empty or not manifest_masked.get("observation_id", pd.Series(dtype=str)).duplicated().any())
missing_images = sum(not resolve_project_path(p).exists() for p in metadata.get("image_path", [])) if "image_path" in metadata else "NA"
add_check("Images selectionnees presentes sur disque", missing_images, missing_images == 0)
add_check("Images avec prediction CNN", cnn_ok, cnn_ok == retained_images)
add_check("Images avec sortie moteur", engine_images, engine_images == cnn_ok)
add_check("Dates valides", int(metadata["date_observation"].isna().sum()) if "date_observation" in metadata else "NA", "date_observation" in metadata and metadata["date_observation"].notna().all())
add_check("Regions autorisees", sorted(set(metadata.get("region", [])) - ALLOWED_REGIONS) if "region" in metadata else "NA", "region" in metadata and set(metadata["region"].dropna()) <= ALLOWED_REGIONS)
add_check("Scores compris entre 0 et 1", int((~compat["score_cnn"].between(0, 1)).sum()) if "score_cnn" in compat else "NA", "score_cnn" in compat and compat["score_cnn"].between(0, 1).all())
add_check("Hors litterature -> Indetermine", int(((compat["niveau_preuve"] == "hors_periode_litterature") & (compat["compatibilite"] != "Indetermine")).sum()) if not compat.empty else "NA", compat.empty or not ((compat["niveau_preuve"] == "hors_periode_litterature") & (compat["compatibilite"] != "Indetermine")).any())
add_check("Sans bulletin -> Indetermine", int(((compat["niveau_preuve"] == "sans_bulletin") & (compat["compatibilite"] != "Indetermine")).sum()) if not compat.empty else "NA", compat.empty or not ((compat["niveau_preuve"] == "sans_bulletin") & (compat["compatibilite"] != "Indetermine")).any())
add_check("Valeurs de verdict autorisees", sorted(set(compat.get("compatibilite", [])) - ALLOWED_VERDICTS) if not compat.empty else "NA", compat.empty or set(compat["compatibilite"].dropna()) <= ALLOWED_VERDICTS)
query_bad = sorted(set().union(*(set(item) for item in query_records)) & TRUTH_LEAKAGE_FIELDS) if query_records else []
add_check("Absence de vraie classe dans la requete moteur", query_bad, not query_bad)
display(pd.DataFrame(checks))

In [ ]:
def sample_case(condition, label, n=1):
    subset = compat[condition].head(n).copy()
    if subset.empty:
        return pd.DataFrame()
    subset.insert(0, "type_cas", label)
    return subset[["type_cas", "image_id", "date_observation", "region", "classe_cnn_predite", "score_cnn", "agent_referentiel", "agent_present_bulletin", "niveau_preuve", "niveau_risque", "compatibilite", "decision_reason"]]

if not compat.empty:
    samples = pd.concat([
        sample_case(compat["compatibilite"].eq("Coherent") & compat["niveau_preuve"].eq("direct"), "Coherent direct"),
        sample_case(compat["compatibilite"].eq("Non_corrobore"), "Non_corrobore"),
        sample_case(compat["niveau_preuve"].eq("hors_periode_litterature"), "Indetermine hors litterature"),
        sample_case(compat["niveau_preuve"].eq("sans_bulletin"), "Indetermine sans bulletin"),
        sample_case(compat["niveau_preuve"].eq("hote_associe"), "Hote associe"),
        sample_case(compat["classe_cnn_predite"].eq("tomato_healthy"), "Healthy"),
    ], ignore_index=True)
    display(samples)

# Partie F - Tableau de bord final et limites

In [ ]:
dashboard_rows = [
    {"indicateur": "PDF collectes", "valeur": len(corpus)},
    {"indicateur": "PDF exploitables", "valeur": int(corpus.get("corpus_exploitable", pd.Series(dtype=str)).astype(str).str.lower().isin(["true","1","oui"]).sum()) if "corpus_exploitable" in corpus else 0},
    {"indicateur": "Bulletins structures", "valeur": len(bulletins)},
    {"indicateur": "Observations BSV", "valeur": len(observations)},
    {"indicateur": "Observations tomate/solanacees", "valeur": len(synthese_tomate)},
    {"indicateur": "Bioagresseurs tomate distincts", "valeur": synthese_tomate["bioagresseur"].nunique() if "bioagresseur" in synthese_tomate else 0},
    {"indicateur": "Classes CNN couvertes", "valeur": int(coverage.get("classe_conservee", pd.Series(dtype=bool)).astype(str).str.lower().isin(["true","1","oui"]).sum()) if "classe_conservee" in coverage else ""},
    {"indicateur": "Images test initiales", "valeur": initial_images},
    {"indicateur": "Images retenues", "valeur": retained_images},
    {"indicateur": "Taux de selection (%)", "valeur": pct(retained_images, initial_images)},
    {"indicateur": "Images rejetees", "valeur": rejected_images},
    {"indicateur": "Cas hors saison manifest", "valeur": hors_saison},
    {"indicateur": "Inferences CNN reussies", "valeur": cnn_ok},
    {"indicateur": "Score Top-1 median", "valeur": round(top1_cnn["score"].median(), 4) if "top1_cnn" in globals() and not top1_cnn.empty else ""},
    {"indicateur": "Predictions evaluees moteur", "valeur": len(compat)},
]
for verdict in ["Coherent", "Non_corrobore", "Indetermine"]:
    dashboard_rows.append({"indicateur": verdict, "valeur": int((compat["compatibilite"] == verdict).sum()) if "compatibilite" in compat else 0})
dashboard_rows.append({"indicateur": "Taux decidabilite Top-1 (%)", "valeur": pct(top1_engine["compatibilite"].isin(["Coherent", "Non_corrobore"]).sum(), len(top1_engine)) if "top1_engine" in globals() and not top1_engine.empty else ""})
dashboard_rows.append({"indicateur": "Risque BSV renseigne", "valeur": int(compat["niveau_risque"].notna().sum()) if "niveau_risque" in compat else 0})
dashboard = pd.DataFrame(dashboard_rows)
display(dashboard)

try:
    main_rejects = value_table(rejected, "raison_rejet").head(3) if not rejected.empty and "raison_rejet" in rejected else pd.DataFrame()
    ind_reasons = value_table(compat[compat["compatibilite"] == "Indetermine"], "niveau_preuve").head(3) if not compat.empty else pd.DataFrame()
    top1_verdicts = value_table(top1_engine, "compatibilite") if "top1_engine" in globals() and not top1_engine.empty else pd.DataFrame()
    print(f"Le corpus contient {len(bulletins)} bulletins BSV structures et {len(observations)} observations phytosanitaires, dont {len(synthese_tomate)} lignes tomate ou solanacees.")
    print(f"Sur {initial_images} images initiales, {retained_images} ont ete retenues, soit {pct(retained_images, initial_images)} %.")
    if not main_rejects.empty:
        print("Les principales causes de rejet sont: " + ", ".join(main_rejects.iloc[:,0].astype(str).tolist()) + ".")
    if "top1_cnn" in globals() and not top1_cnn.empty:
        print(f"Le CNN a produit une prediction pour {top1_cnn['observation_id'].nunique()} images ; le score Top-1 median est {top1_cnn['score'].median():.4f}.")
    if not top1_verdicts.empty:
        d = dict(zip(top1_verdicts.iloc[:,0], top1_verdicts["pct"]))
        print(f"Pour les predictions Top-1, le moteur produit {d.get('Coherent', 0)} % de Coherent, {d.get('Non_corrobore', 0)} % de Non_corrobore et {d.get('Indetermine', 0)} % d'Indetermine.")
    if not ind_reasons.empty:
        print("Les principales causes d'indetermination sont: " + ", ".join(ind_reasons.iloc[:,0].astype(str).tolist()) + ".")
    print("Ces resultats decrivent le fonctionnement et la couverture du pipeline. Ils ne constituent pas encore une mesure de la performance du moteur.")
except Exception as exc:
    print(f"Resume narratif indisponible: {exc}")

## Limites de l'analyse descriptive

- le jeu experimental est cible sur les classes couvertes par les BSV ;
- les dates et regions sont attribuees experimentalement ;
- les BSV decrivent un contexte regional et non l'etat biologique de l'image ;
- certaines classes CNN sont peu ou non couvertes ;
- un verdict `Non_corrobore` ne signifie pas que la prediction CNN est fausse ;
- l'analyse descriptive ne mesure pas encore l'exactitude du moteur.

## Prochaine etape

Evaluer l'association entre la justesse du CNN et les verdicts du moteur BSV
avec `src/phase3_evaluate.py`, puis, si necessaire, avec des tableaux de
contingence, le MCC, un test du chi-2 ou de Fisher, des odds ratios et des
intervalles de confiance bootstrap.